# Comprehensive Model Evaluation — iteris

Cross-cutting evaluation of every model produced by this project: U-Net baselines
(**LiteUNet**, **AttentionResUNet**) and DRL refiners (**DuelingDDQN** — discrete
contour-sector actions, **TD3** — continuous contour-sector actions) across both
datasets (**CAMUS**: LV_endo / LV_epi / LA, **BRISC**: glioma / meningioma /
pituitary / tumor).

This notebook does **not** hardcode any results. It reads whatever run-output
files you drop into `results/` and computes every table/plot/verdict from them.
If a section has no matching files it prints what it's missing instead of
fabricating numbers.

## Expected inputs

Drop your run outputs anywhere under `results/` (subfolders are fine — files are
identified by their JSON **content**, not their path, so naming collisions across
runs in different folders are harmless). Two shapes are recognized automatically:

1. **U-Net summaries** — `iteris.evaluation.save_summary_json` output
   (`<dataset><suffix>_summary.json`). Identified by the presence of a
   `test_dice` dict key.
2. **DRL test-set summaries** — `iteris.drl_reeval.reeval_checkpoint` output
   (`<dataset>_<agent>_c<class>_reeval_test_results.json`), or any dict produced
   by `iteris.refinement_viz.evaluate_testset` with a `dataset` / `agent_type` /
   `class_name` field added. Identified by the presence of `init_dice_mean` +
   `final_dice_mean` keys.

Optional, for real significance testing (Section 7): a **per-sample CSV** per
DRL run with one row per test patient and at least `init_dice`, `final_dice`
(and ideally `value_floored_dice`) columns — e.g. exported from the `replays`
list returned by `evaluate_testset`/`build_replays`. Without this, Section 7
reports the aggregate deltas only and explains what's missing for a real test.

## Sections
1. Ingestion
2. Master tidy comparison table
3. U-Net baseline landscape (Lite vs Attention — the DRL headroom ceiling)
4. Discrete vs continuous action space (DuelingDDQN vs TD3)
5. Per-class / per-dataset learning behaviour
6. Most-improved DRL agent ranking
7. Statistical significance (Wilcoxon signed-rank + Bonferroni)
8. Export & auto-generated summary


In [ ]:
import json, glob, os
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3


In [ ]:
# Configuration
# Run from notebooks/evaluation/ (repo-relative) or from the repo root -- try both.
_candidates = [Path('../../results'), Path('results'), Path('../results')]
RESULTS_DIR = next((p for p in _candidates if p.exists()), _candidates[0])
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'[config] RESULTS_DIR = {RESULTS_DIR.resolve()}')

# agent_type (as stored in cfg / saved JSON) -> action space + display name
AGENT_META = {
    'TD3':      dict(action_space='continuous', display='TD3'),
    'DUELING':  dict(action_space='discrete',   display='DuelingDDQN'),
    'DDPG':     dict(action_space='continuous', display='DDPG (archived global)'),
    'DQN':      dict(action_space='discrete',   display='DQN (archived global)'),
}

# internal cfg['model'] key -> display name (matches iteris.evaluation._model_names)
UNET_META = {
    'lite_unet':    dict(display='LiteUNet'),
    'attn_resunet': dict(display='AttentionResUNet'),
    'LiteUNet':     dict(display='LiteUNet'),
    'AttentionResUNet': dict(display='AttentionResUNet'),
}

# Known class taxonomy, for stable ordering in tables/plots (not enforced --
# any class name found in the data is included even if absent here).
DATASET_CLASS_ORDER = {
    'CAMUS': ['LV_endo', 'LV_epi', 'LA'],
    'BRISC': ['glioma', 'meningioma', 'pituitary', 'tumor'],
}


## 1. Ingestion\n\nWalk every `*.json` under `RESULTS_DIR`, sniff its shape, and route it to the U-Net or DRL parser. Anything unrecognized is reported, not silently dropped.

In [ ]:
def _find_json_files(root: Path):
    return sorted(root.rglob('*.json')) if root.exists() else []

def _load_json(path: Path):
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f'  [skip] {path}: could not parse ({e})')
        return None

def parse_unet_summary(d, path):
    # iteris.evaluation.save_summary_json shape -> long rows, one per class.
    rows = []
    dataset = d.get('dataset', '?')
    model_key = d.get('model', '?')
    model = UNET_META.get(model_key, {}).get('display', model_key)
    for cls, dice in d.get('test_dice', {}).items():
        rows.append(dict(
            source=str(path), dataset=dataset, model=model, model_family='UNet',
            action_space='n/a', class_name=cls,
            dice=dice,
            iou=d.get('test_iou', {}).get(cls),
            biou=d.get('test_biou', {}).get(cls),
            precision=d.get('test_precision', {}).get(cls),
            sensitivity=d.get('test_sensitivity', {}).get(cls),
            hd95=d.get('test_hd95', {}).get(cls),
            hd95geo=d.get('test_hd95geo_px', {}).get(cls),
            msd=d.get('test_msd_px', {}).get(cls),
            best_val_dice=d.get('best_val_dice'),
            label_frac=d.get('label_frac'),
        ))
    return rows

def parse_drl_summary(d, path):
    # iteris.refinement_viz.evaluate_testset (+ reeval wrapper) shape -> one row.
    agent_type = str(d.get('agent_type', '?')).upper()
    meta = AGENT_META.get(agent_type, dict(action_space='unknown', display=agent_type))
    return [dict(
        source=str(path),
        dataset=d.get('dataset', '?'),
        class_name=d.get('class_name', '?'),
        target_class=d.get('target_class'),
        model=meta['display'], model_family='DRL', action_space=meta['action_space'],
        agent_type=agent_type,
        init_dice=d.get('init_dice_mean'),
        final_dice=d.get('final_dice_mean'),
        best_dice_ceiling=d.get('best_dice_mean'),
        value_floored_dice=d.get('value_floored_dice_mean'),
        delta_dice=d.get('delta_dice_mean'),
        value_floored_delta=d.get('value_floored_delta_mean'),
        final_hd95=d.get('final_hd95_mean'),
        init_hd95=d.get('init_hd95_mean'),
        final_iou=d.get('final_iou_mean'),
        init_iou=d.get('init_iou_mean'),
        delta_iou=d.get('delta_iou_mean'),
        final_biou=d.get('final_biou_mean'),
        init_biou=d.get('init_biou_mean'),
        delta_biou=d.get('delta_biou_mean'),
        final_precision=d.get('final_precision_mean'),
        final_sensitivity=d.get('final_sensitivity_mean'),
        final_msd=d.get('final_msd_mean'),
        init_msd=d.get('init_msd_mean'),
        n_refinable=d.get('n_refinable'), n_total=d.get('n_total'),
        refinable_frac=d.get('refinable_frac'),
        value_floored_dice_refinable=d.get('value_floored_dice_refinable_mean'),
        value_floored_delta_refinable=d.get('value_floored_delta_refinable_mean'),
        routed_dice=d.get('routed_dice_mean'), routed_delta=d.get('routed_delta_mean'),
        reeval=d.get('reeval', False), source_checkpoint=d.get('source_checkpoint'),
    )]

unet_rows, drl_rows, unrecognized = [], [], []
for path in _find_json_files(RESULTS_DIR):
    d = _load_json(path)
    if d is None:
        continue
    if isinstance(d, dict) and 'test_dice' in d:
        unet_rows += parse_unet_summary(d, path)
    elif isinstance(d, dict) and {'init_dice_mean', 'final_dice_mean'} <= d.keys():
        drl_rows += parse_drl_summary(d, path)
    else:
        unrecognized.append(path)

unet_df = pd.DataFrame(unet_rows)
drl_df = pd.DataFrame(drl_rows)

print(f'[ingestion] U-Net summary rows: {len(unet_df)}  (files: {unet_df["source"].nunique() if len(unet_df) else 0})')
print(f'[ingestion] DRL summary rows:   {len(drl_df)}  (files: {drl_df["source"].nunique() if len(drl_df) else 0})')
if unrecognized:
    print(f'[ingestion] {len(unrecognized)} JSON file(s) matched neither shape, skipped:')
    for p in unrecognized:
        print(f'    - {p}')

if unet_df.empty and drl_df.empty:
    print(f"""
No result files found under {RESULTS_DIR.resolve()}.
Drop your *_summary.json (U-Net) and *_reeval_test_results.json / evaluate_testset-shaped
JSON (DRL) files anywhere under this folder (subfolders OK), then re-run this notebook.
""")


In [ ]:
# Optional per-sample CSVs for Section 7 (real paired significance testing).
# Any CSV with an 'init_dice' + 'final_dice' column (or 'dice_<class>' columns from
# iteris.evaluation.evaluate_test_set) is picked up; tag it with dataset/class/agent
# parsed from its filename since per-sample exports don't carry that as JSON metadata.
per_sample_csvs = sorted(RESULTS_DIR.rglob('*.csv')) if RESULTS_DIR.exists() else []
per_sample_frames = {}
for p in per_sample_csvs:
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print(f'  [skip] {p}: {e}')
        continue
    if {'init_dice', 'final_dice'} <= set(df.columns):
        per_sample_frames[str(p)] = df
    elif any(c.startswith('dice_') for c in df.columns):
        per_sample_frames[str(p)] = df  # U-Net per-patient test_scores.csv
print(f'[ingestion] per-sample CSVs usable for paired testing: {len(per_sample_frames)}')
for k in per_sample_frames:
    print(f'    - {k}')


## 2. Master tidy comparison table\n\nOne row per (dataset, class, model), Dice as the common column, tagged by `model_family` (UNet / DRL) and `action_space` (n/a / discrete / continuous) so every later section is just a filter + groupby over this one frame.

In [ ]:
master_rows = []
if not unet_df.empty:
    for _, r in unet_df.iterrows():
        master_rows.append(dict(dataset=r['dataset'], class_name=r['class_name'],
                                 model=r['model'], model_family='UNet', action_space='n/a',
                                 dice=r['dice'], hd95=r['hd95'], iou=r['iou']))
if not drl_df.empty:
    for _, r in drl_df.iterrows():
        # 'final_dice' can be worse than init (no learned STOP drift, see TD3 note in
        # docs) -- value_floored_dice is the honest deployable number; fall back to
        # final_dice only if it's missing.
        deploy_dice = r['value_floored_dice'] if pd.notna(r.get('value_floored_dice')) else r['final_dice']
        master_rows.append(dict(dataset=r['dataset'], class_name=r['class_name'],
                                 model=r['model'], model_family='DRL', action_space=r['action_space'],
                                 dice=deploy_dice, hd95=r['final_hd95'], iou=r['final_iou']))

master_df = pd.DataFrame(master_rows)
if not master_df.empty:
    master_df = master_df.sort_values(['dataset', 'class_name', 'model_family', 'model']).reset_index(drop=True)
master_df


## 3. U-Net baseline landscape

`LiteUNet` is the deliberately weak baseline the DRL agents refine (chosen because it has
headroom); `AttentionResUNet` is the strong upper-bound competitor. The gap between them,
per class, is the ceiling a DRL agent could theoretically reach by fully closing the
representation/information gap — a useful reference when judging whether a DRL Dice gain
is meaningful or just noise.

In [ ]:
if unet_df.empty:
    print('No U-Net summaries loaded -- skipping. Drop *_summary.json files under results/.')
else:
    pivot = unet_df.pivot_table(index=['dataset', 'class_name'], columns='model', values='dice')
    if {'LiteUNet', 'AttentionResUNet'} <= set(pivot.columns):
        pivot['headroom_to_attn'] = pivot['AttentionResUNet'] - pivot['LiteUNet']
    display(pivot)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    plot_df = unet_df.copy()
    plot_df['label'] = plot_df['dataset'] + ' / ' + plot_df['class_name']
    models = sorted(plot_df['model'].unique())
    labels = sorted(plot_df['label'].unique())
    width = 0.8 / max(len(models), 1)
    for i, m in enumerate(models):
        sub = plot_df[plot_df['model'] == m].set_index('label')['dice'].reindex(labels)
        ax.bar(np.arange(len(labels)) + i * width, sub.values, width, label=m)
    ax.set_xticks(np.arange(len(labels)) + width * (len(models) - 1) / 2)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('Test Dice'); ax.set_title('U-Net baselines by dataset / class'); ax.legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / 'unet_baseline_landscape.png'); plt.show()


## 4. Discrete vs continuous action space

`DuelingDDQN` acts over discrete contour-sector edits (with a learned STOP action);
`TD3` acts over continuous per-sector displacements (no learned STOP -- converge-and-hold
+ a `fail_thresh` safety net instead, see `iteris/drl_reeval/re_eval_td3.py`). That
asymmetry matters for a fair comparison: TD3 can drift past its own peak (`best_dice_ceiling`
vs `final_dice` gap) in a way DuelingDDQN structurally can't, so drift is reported
explicitly rather than folded into a single number.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping. Drop *_reeval_test_results.json files under results/.')
else:
    drl_df['drift'] = drl_df['best_dice_ceiling'] - drl_df['final_dice']
    drl_df['deploy_delta'] = drl_df['value_floored_delta'].fillna(drl_df['delta_dice'])

    agg = drl_df.groupby('action_space').agg(
        n_runs=('model', 'size'),
        mean_deploy_delta=('deploy_delta', 'mean'),
        std_deploy_delta=('deploy_delta', 'std'),
        win_rate=('deploy_delta', lambda s: float((s > 0).mean())),
        mean_drift=('drift', 'mean'),
        mean_final_hd95=('final_hd95', 'mean'),
    ).round(4)
    display(agg)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    drl_df.boxplot(column='deploy_delta', by='action_space', ax=axes[0])
    axes[0].axhline(0, color='r', ls='--', lw=1)
    axes[0].set_title('Deployable delta-Dice by action space'); axes[0].set_ylabel('delta-Dice (value-floored)')
    drl_df.boxplot(column='drift', by='action_space', ax=axes[1])
    axes[1].set_title('Drift: best-seen minus final (deploy) Dice'); axes[1].set_ylabel('delta-Dice')
    fig.suptitle(''); plt.tight_layout()
    fig.savefig(FIG_DIR / 'discrete_vs_continuous.png'); plt.show()


## 5. Per-class / per-dataset learning behaviour

A dataset-class-agent cell shows genuine positive **learning** (not a collapse to
"do-nothing at baseline") when the deployable delta (`value_floored_delta`, GT-free
selection) is clearly positive. Cells at approximately 0 indicate the STOP-at-baseline
collapse described in the project's PBRS fix notes; negative cells indicate the agent
is actively hurting the baseline.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    heat = drl_df.pivot_table(index=['dataset', 'class_name'], columns='model',
                               values='deploy_delta')
    try:
        display(heat.style.background_gradient(cmap='RdYlGn', axis=None, vmin=-0.05, vmax=0.05))
    except Exception:
        display(heat)

    fig, ax = plt.subplots(figsize=(1.6 * max(len(heat.columns), 2) + 2, 0.5 * len(heat) + 2))
    im = ax.imshow(heat.values, cmap='RdYlGn', vmin=-0.05, vmax=0.05, aspect='auto')
    ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(heat.index))); ax.set_yticklabels([f'{d}/{c}' for d, c in heat.index])
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f'{v:+.3f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, label='Deployable delta-Dice vs U-Net baseline')
    ax.set_title('Learning behaviour: deployable delta-Dice by dataset/class x agent')
    fig.tight_layout(); fig.savefig(FIG_DIR / 'learning_behaviour_heatmap.png'); plt.show()

    print()
    print('Best-learning (dataset, class) per agent (highest deployable delta-Dice):')
    for m in drl_df['model'].unique():
        sub = drl_df[drl_df['model'] == m].sort_values('deploy_delta', ascending=False)
        if not sub.empty:
            top = sub.iloc[0]
            print(f'  {m:>14s}: {top["dataset"]}/{top["class_name"]}  delta-Dice={top["deploy_delta"]:+.4f}')


## 6. Most-improved DRL agent ranking

Ranks every (agent, dataset, class) run by deployable delta-Dice. Read this alongside
Section 3's U-Net headroom and the caveats below -- a run with near-zero headroom
(strong baseline already near-ceiling) improving little is not the same failure mode
as a run with real headroom that the agent still couldn't reach; see the project's
representation/information/optimization-cap analysis referenced in Section 8.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    ranked = drl_df.sort_values('deploy_delta', ascending=False)[
        ['dataset', 'class_name', 'model', 'action_space', 'init_dice', 'final_dice',
         'value_floored_dice', 'deploy_delta', 'best_dice_ceiling', 'drift', 'refinable_frac']
    ].reset_index(drop=True)
    display(ranked)

    per_agent = drl_df.groupby('model').agg(
        n_runs=('deploy_delta', 'size'),
        mean_deploy_delta=('deploy_delta', 'mean'),
        median_deploy_delta=('deploy_delta', 'median'),
        n_positive=('deploy_delta', lambda s: int((s > 0).sum())),
    ).sort_values('mean_deploy_delta', ascending=False).round(4)
    display(per_agent)

    if not per_agent.empty:
        winner = per_agent.index[0]
        print()
        print(f'>>> Most successful DRL agent by mean deployable delta-Dice across all loaded runs: '
              f'{winner}  (mean delta-Dice={per_agent.loc[winner, "mean_deploy_delta"]:+.4f}, '
              f'{per_agent.loc[winner, "n_positive"]}/{per_agent.loc[winner, "n_runs"]} runs beat baseline)')


## 7. Statistical significance (Wilcoxon signed-rank + Bonferroni)

Requires **per-sample paired** data (same test patient's Dice before/after refinement).
Aggregate JSON summaries alone (Sections 1-6) cannot support a real hypothesis test —
they're already means. If you exported per-sample CSVs (see the notebook intro), each
one is tested here; otherwise this section explains exactly what's missing.

In [ ]:
if not per_sample_frames:
    print("""No per-sample CSVs found -- cannot run a real paired significance test.
To enable this section, export one CSV per DRL run with per-test-patient
'init_dice' and 'final_dice' (ideally also 'value_floored_dice') columns, e.g. from
the `replays` list `iteris.refinement_viz.build_replays` / `evaluate_testset` computes
internally (r['init_dice'], r['final_dice'], r['value_floored_dice'] per sample) --
drop them anywhere under results/ and re-run.""")
else:
    sig_rows = []
    for path, df in per_sample_frames.items():
        if {'init_dice', 'final_dice'} <= set(df.columns):
            paired = df[['init_dice', 'final_dice']].dropna()
            if len(paired) < 5:
                print(f'  [skip] {path}: only {len(paired)} paired samples, too few for Wilcoxon')
                continue
            stat, p = stats.wilcoxon(paired['final_dice'], paired['init_dice'])
            sig_rows.append(dict(source=path, n=len(paired),
                                  mean_delta=(paired['final_dice'] - paired['init_dice']).mean(),
                                  wilcoxon_stat=stat, p_raw=p))
    if sig_rows:
        sig_df = pd.DataFrame(sig_rows)
        # Bonferroni across every test run in this notebook
        sig_df['p_bonferroni'] = np.minimum(sig_df['p_raw'] * len(sig_df), 1.0)
        sig_df['significant_at_0.05'] = sig_df['p_bonferroni'] < 0.05
        display(sig_df)
    else:
        print('No CSV had a usable init_dice/final_dice pair with n>=5.')


## 8. Export & auto-generated summary

Saves the master table and figures, then prints a summary built entirely from the
computed values above (nothing here is pre-written prose -- if you re-run this
notebook against different result files, the verdicts below change with the data).

**Context for interpreting a "DRL didn't beat baseline" result** (from this project's
own diagnostics, `docs/CONTEXT.md`): three separate, non-overlapping caps can each
produce that outcome, and only the last is an optimization problem the training
recipe can fix -- (a) **representation cap**: the deployed mask is a smoothed
periodic-spline approximation of the largest connected component, never the raw
U-Net mask, so even the do-nothing policy pays a tax vs. the U-Net benchmark;
(b) **information cap**: a GT-blind refiner has no privileged signal the U-Net
lacked, so image-ambiguous residual errors may be genuinely unfixable without a
learned shape prior; (c) **optimization cap**: reward shaping / STOP-timing /
training budget -- the only cap that more training or reward tuning addresses.
Run `iteris.diagnostics.headroom_report` (oracle-contour-ceiling vs baseline) to
tell these apart for any run showing near-zero or negative delta-Dice here.

In [ ]:
if not master_df.empty:
    out_csv = RESULTS_DIR / 'master_comparison.csv'
    master_df.to_csv(out_csv, index=False)
    print(f'[export] master comparison table -> {out_csv}')

print()
print('=' * 70)
print('SUMMARY (auto-generated from loaded results)')
print('=' * 70)

if unet_df.empty and drl_df.empty:
    print('No result files loaded yet -- nothing to summarize. See Section 1.')
else:
    if not unet_df.empty:
        best_unet = unet_df.sort_values('dice', ascending=False).iloc[0]
        print(f'- Strongest U-Net baseline observed: {best_unet["model"]} on '
              f'{best_unet["dataset"]}/{best_unet["class_name"]} (Dice={best_unet["dice"]:.4f}).')
    if not drl_df.empty:
        agent_summary = drl_df.groupby('action_space')['deploy_delta'].mean()
        for space, val in agent_summary.items():
            print(f'- Mean deployable delta-Dice, {space} action space: {val:+.4f}')
        if len(agent_summary) >= 2:
            better_space = agent_summary.idxmax()
            print(f'  -> {better_space} action space shows the larger average deployable '
                  f'gain across loaded runs.')
        per_agent = drl_df.groupby('model')['deploy_delta'].mean().sort_values(ascending=False)
        print(f'- Best-performing DRL agent overall: {per_agent.index[0]} '
              f'(mean delta-Dice={per_agent.iloc[0]:+.4f}).')
        best_cell = drl_df.sort_values('deploy_delta', ascending=False).iloc[0]
        print(f'- Single best (agent, dataset, class) result: {best_cell["model"]} on '
              f'{best_cell["dataset"]}/{best_cell["class_name"]} '
              f'(delta-Dice={best_cell["deploy_delta"]:+.4f}).')
        n_beat = int((drl_df['deploy_delta'] > 0).sum())
        print(f'- {n_beat}/{len(drl_df)} loaded DRL runs beat their U-Net baseline '
              f'(deployable, value-floored comparison).')
print('=' * 70)
